<a href="https://colab.research.google.com/github/Frankieche24/Python_exercise1/blob/main/python_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd

loans = pd.read_csv("kiva_loans.csv")
print(loans.shape)
loans.head()

(15439, 20)


,id,funded_amount,loan_amount,activity,sector,use,country_code,country,region,currency,partner_id,posted_time,disbursed_time,funded_time,term_in_months,lender_count,tags,borrower_genders,repayment_interval,date
0,653051,300.0,300.0,Fruits & Vegetables,Food,"To buy seasonal, fresh fruits to sell.",PK,Pakistan,Lahore,PKR,247.0,2014-01-01 06:12:39+00:00,2013-12-17 08:00:00+00:00,2014-01-02 10:06:32+00:00,12.0,12.0,NaN,female,irregular,2014-01-01
1,653053,575.0,575.0,Rickshaw,Transportation,to repair and maintain the auto rickshaw used ...,PK,Pakistan,Lahore,PKR,247.0,2014-01-01 06:51:08+00:00,2013-12-17 08:00:00+00:00,2014-01-02 09:17:23+00:00,11.0,14.0,NaN,"female, female",irregular,2014-01-01
2,653068,150.0,150.0,Transportation,Transportation,To repair their old cycle-van and buy another ...,IN,India,Maynaguri,INR,334.0,2014-01-01 09:58:07+00:00,2013-12-17 08:00:00+00:00,2014-01-01 16:01:36+00:00,43.0,6.0,"user_favorite, user_favorite",female,bullet,2014-01-01
3,653063,200.0,200.0,Embroidery,Arts,to purchase an embroidery machine and a variet...,PK,Pakistan,Lahore,PKR,247.0,2014-01-01 08:03:11+00:00,2013-12-24 08:00:00+00:00,2014-01-01 13:00:00+00:00,11.0,8.0,NaN,female,irregular,2014-01-01
4,653084,400.0,400.0,Milk Sales,Food,to purchase one buffalo.,PK,Pakistan,Abdul Hakeem,PKR,245.0,2014-01-01 11:53:19+00:00,2013-12-17 08:00:00+00:00,2014-01-01 19:18:51+00:00,14.0,16.0,NaN,female,monthly,2014-01-01


# Question 1: Which countries have the most large agricultural loans for seeds or fertilizer?


In [4]:
farm = loans[
    (loans["sector"] == "Agriculture")
    & (loans["loan_amount"] > 1000)
    & (loans["use"].str.contains("fertilizer|seed", case=False, na=False))
]

farm_by_country = (
    farm.groupby("country")
        .agg(n_loans=("id", "count"), avg_loan_amount=("loan_amount", "mean"))
        .sort_values("n_loans", ascending=False)
        .head(10)
        .round(0)
)
print(len(farm), "loans match")
farm_by_country

373 loans match


,n_loans,avg_loan_amount
country,,
Kenya,167,3989.0
Tajikistan,31,1602.0
Cambodia,27,1246.0
Ecuador,21,1410.0
India,18,3376.0
El Salvador,13,1442.0
Ukraine,11,2518.0
Senegal,10,2348.0
Peru,10,1990.0


# Question 2: Which sectors take the longest to get fully funded?


In [5]:
timed = loans.dropna(subset=["posted_time", "funded_time"]).copy()

# Convert time columns to datetime objects before subtraction
timed["posted_time"] = pd.to_datetime(timed["posted_time"])
timed["funded_time"] = pd.to_datetime(timed["funded_time"])

timed["days_to_fund"] = (timed["funded_time"] - timed["posted_time"]).dt.total_seconds() / 86400
timed = timed[timed["days_to_fund"] >= 0]

speed = (
    timed.groupby("sector")["days_to_fund"]
         .agg(n_loans="count", median_days="median", mean_days="mean")
         .sort_values("median_days", ascending=False)
         .round(1)
)
speed

,n_loans,median_days,mean_days
sector,,,
Housing,801,28.6,26.8
Personal Use,326,8.7,14.6
Entertainment,20,8.4,15.0
Clothing,835,7.3,13.2
Transportation,504,6.6,13.0
Construction,189,6.6,12.0
Retail,3011,6.3,13.0
Services,1132,5.0,11.9
Food,3402,3.8,10.9


# Question 3: Do women only borrower groups borrow differently from men only and mixed groups?


In [6]:
g = loans.dropna(subset=["borrower_genders"]).copy()

# borrower_genders looks like "female, female, male"
# removing "female" first stops it from matching the word "male"
g["group_type"] = g["borrower_genders"].apply(
    lambda s: "women only" if "male" not in s.replace("female", "")
    else "men only" if "female" not in s
    else "mixed"
)

by_group = (
    g.groupby("group_type")
     .agg(n_loans=("id", "count"),
          median_loan=("loan_amount", "median"),
          mean_lenders=("lender_count", "mean"),
          mean_term_months=("term_in_months", "mean"))
     .sort_values("n_loans", ascending=False)
     .round(1)
)
by_group

,n_loans,median_loan,mean_lenders,mean_term_months
group_type,,,,
women only,10948,450.0,22.6,13.7
men only,3772,750.0,28.9,16.8
mixed,550,3450.0,87.6,9.2


# Question 4: Which activities are most often not fully funded?



In [7]:
loans["fully_funded"] = loans["funded_amount"] >= loans["loan_amount"]

activities = (
    loans.groupby("activity")
         .agg(n_loans=("id", "count"),
              share_not_funded=("fully_funded", lambda x: 1 - x.mean()))
)

# only activities with at least 500 loans, so the shares are not based on a handful of loans
activities = activities[activities["n_loans"] >= 500].sort_values("share_not_funded", ascending=False)
print(f"All loans: {1 - loans['fully_funded'].mean():.3f} not fully funded")
print("Lowest three activities:")
print(activities.tail(3).round(3))
activities.head(10).round(3)

All loans: 0.027 not fully funded
Lowest three activities:
                        n_loans  share_not_funded
activity                                         
Food Production/Sales       618             0.010
General Store              1419             0.008
Higher education costs      730             0.000


,n_loans,share_not_funded
activity,,
Personal Housing Expenses,815,0.054
Agriculture,714,0.038
Farming,1346,0.035
Clothing Sales,578,0.026
Retail,767,0.021
Food Production/Sales,618,0.010
General Store,1419,0.008
Higher education costs,730,0.000


# Question 5: In East Africa, how does loan size relate to repayment interval and loan term?


In [8]:
east = loans[loans["country"].isin(["Kenya", "Uganda", "Tanzania", "Rwanda"])].copy()

east["size_band"] = east["loan_amount"].apply(
    lambda x: "1 small (<300)" if x < 300
    else "2 medium (300-999)" if x < 1000
    else "3 large (1000+)"
)

size_term = (
    east.groupby(["size_band", "repayment_interval"])
        .agg(n_loans=("id", "count"), avg_term_months=("term_in_months", "mean"))
        .reset_index()
        .sort_values(["size_band", "avg_term_months"], ascending=[True, False])
        .round(1)
)
size_term

,size_band,repayment_interval,n_loans,avg_term_months
0,1 small (<300),bullet,24,42.0
2,1 small (<300),monthly,461,17.2
1,1 small (<300),irregular,536,8.3
3,1 small (<300),weekly,4,1.8
4,2 medium (300-999),bullet,21,38.6
6,2 medium (300-999),monthly,513,12.6
5,2 medium (300-999),irregular,462,11.1
7,2 medium (300-999),weekly,4,5.8
10,3 large (1000+),monthly,149,13.2
9,3 large (1000+),irregular,145,12.3


# Question 6: Which short loans attracted the most lenders?



In [9]:
short = loans[(loans["term_in_months"] <= 6) & (loans["lender_count"] >= 50)]

top_short = (
    short.sort_values(["lender_count", "loan_amount"], ascending=[False, True])
         [["country", "sector", "activity", "loan_amount", "lender_count", "term_in_months"]]
         .head(10)
)
print(len(short), "short loans have at least 50 lenders")
display(top_short)

short.groupby("sector").size().sort_values(ascending=False).head(5)

72 short loans have at least 50 lenders


,country,sector,activity,loan_amount,lender_count,term_in_months
5528,The Democratic Republic of the Congo,Food,Food Stall,8125.0,273.0,6.0
6957,The Democratic Republic of the Congo,Clothing,Used Clothing,7125.0,249.0,6.0
7458,The Democratic Republic of the Congo,Clothing,Used Shoes,5350.0,175.0,6.0
93,Tanzania,Retail,Charcoal Sales,5775.0,154.0,6.0
6586,The Democratic Republic of the Congo,Food,Food Stall,4600.0,151.0,6.0
1198,Peru,Services,Beauty Salon,5625.0,149.0,6.0
5493,The Democratic Republic of the Congo,Clothing,Used Clothing,3825.0,134.0,6.0
7339,Tanzania,Food,Grocery Store,3800.0,133.0,6.0
7696,The Democratic Republic of the Congo,Clothing,Used Shoes,4625.0,133.0,6.0
3813,Peru,Food,Butcher Shop,3950.0,128.0,6.0


,0
sector,
Food,33
Retail,15
Clothing,11
Arts,4
Agriculture,3
